In [1]:
import os
import random
import numpy as np
import torch

def seed_everything(seed: int = 42):
    # Python
    random.seed(seed)
    
    # Numpy
    np.random.seed(seed)
    
    # PyTorch (CPU)
    torch.manual_seed(seed)
    
    # PyTorch (GPU)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Determinismo (importante!)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Algumas libs usam isso
    os.environ["PYTHONHASHSEED"] = str(seed)
    
    # Para transformers (às vezes ajuda)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    print(f"Seed definida como {seed}")

# Uso
seed_everything(42)

Seed definida como 42


In [2]:
import numpy as np
import os
from datetime import datetime
import pandas as pd
from tqdm import tqdm

import datasets
import evaluate

import torch
import torch.nn as nn

from transformers import AutoTokenizer, BertConfig, BertModel, BertPreTrainedModel

from sklearn.metrics import f1_score, accuracy_score

/home/guilhermelima/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Mapeamento real do dataset de treino (ordenado por frequência, não alfabeticamente)
UPOS_LABELS = ['DET', 'NOUN', 'VERB', 'PUNCT', 'SCONJ', 'ADP', 'ADJ', 'CCONJ', 'ADV', 'PROPN', 'AUX', 'NUM', 'PRON', 'SYM', 'X', 'INTJ']
DEPREL_LABELS = ['det', 'nsubj', 'root', 'obj', 'xcomp', 'punct', 'mark', 'advcl', 'case', 'obl', 'amod', 'conj', 'cc', 'nmod', 'advmod', 'flat:name', 'ccomp', 'cop', 'acl', 'nummod', 'acl:relcl', 'ccomp:speech', 'parataxis', 'csubj', 'aux:pass', 'appos', 'fixed', 'nsubj:pass', 'aux', 'nsubj:outer', 'obl:agent', 'expl:impers', 'expl', 'discourse', 'orphan', 'dislocated', 'flat', 'flat:foreign', 'iobj', 'vocative', 'csubj:outer', 'list', 'reparandum', 'csubj:pass']

DEPREL_LABELS_TO_IDX = {
    i : idx
    for idx, i in enumerate(DEPREL_LABELS)
}

IDX_TO_DEPREL_LABELS = {
    i: j
    for j, i in DEPREL_LABELS_TO_IDX.items()
}


UPOS_LABELS_TO_IDX = {
    i : idx
    for idx, i in enumerate(UPOS_LABELS)
}

IDX_TO_UPOS_LABELS = {
    i: j
    for j, i in UPOS_LABELS_TO_IDX.items()
}


NUM_TRAIN_EPOCHS = 10

# ── Para trocar de modelo, altere apenas estas duas variáveis ──────────────────
# BERTimbau-large : FINETUNED_MODEL_PATH = '.../checkpoint-28743'
#                   TOKENIZER_NAME = 'neuralmind/bert-large-portuguese-cased'
# modernJabutica  : FINETUNED_MODEL_PATH = '.../checkpoint-29480'
#                   TOKENIZER_NAME = 'amadeusai/modernJabuticaBERT-Base-1k'
FINETUNED_MODEL_PATH = '/home/guilhermelima/msc/trainer_output/checkpoint-28743'
TOKENIZER_NAME       = 'amadeusai/modernJabuticaBERT-Base-1k'
# ──────────────────────────────────────────────────────────────────────────────


In [4]:
# model definition
'''class MultiTaskSentencePrediction(BertPreTrainedModel):
    def __init__(self, config, num_xpos_labels, num_deprel_labels):
        super().__init__(config)
        self.num_xpos_labels = num_xpos_labels
        self.num_deprel_labels = num_deprel_labels

        self.bert = BertModel(config)

        self.xpos_classifier = nn.Linear(config.hidden_size, num_xpos_labels)
        self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)

        classifier_dropout = config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        
        self.dropout = nn.Dropout(classifier_dropout)
        self.init_weights()

    def forward(
            self, input_ids, attention_mask=None, token_type_ids=None, 
            xpos_label=None, deprel_label=None 
    ):
        outputs = self.bert(
            input_ids=input_ids, attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        sequence_output = self.dropout(outputs[0])  # [batch_size, seq_len, hidden_size]


        
        # Classificação por token
        logits_xpos = self.xpos_classifier(sequence_output)     # [batch_size, seq_len, num_xpos_labels]
        logits_deprel = self.deprel_classifier(sequence_output)
        


        loss = None
        if xpos_label != None: #and deprel_label != None:
            
            
            loss_fct1 = nn.CrossEntropyLoss(ignore_index=-100)
            loss_fct2 = nn.CrossEntropyLoss(ignore_index=-100)

            loss = loss_fct1(
            logits_xpos.view(-1, self.num_xpos_labels),  # [B * L, num_xpos_labels]
            xpos_label.view(-1)                          # [B * L]
        ) + loss_fct2(
            logits_deprel.view(-1, self.num_deprel_labels),
            deprel_label.view(-1)
        )
        #return (loss, logits_xpos) if loss is not None else (logits_xpos)
        return (loss, logits_xpos, logits_deprel) if loss is not None else (logits_xpos, logits_deprel)'''

'class MultiTaskSentencePrediction(BertPreTrainedModel):\n    def __init__(self, config, num_xpos_labels, num_deprel_labels):\n        super().__init__(config)\n        self.num_xpos_labels = num_xpos_labels\n        self.num_deprel_labels = num_deprel_labels\n\n        self.bert = BertModel(config)\n\n        self.xpos_classifier = nn.Linear(config.hidden_size, num_xpos_labels)\n        self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)\n\n        classifier_dropout = config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob\n        \n        self.dropout = nn.Dropout(classifier_dropout)\n        self.init_weights()\n\n    def forward(\n            self, input_ids, attention_mask=None, token_type_ids=None, \n            xpos_label=None, deprel_label=None \n    ):\n        outputs = self.bert(\n            input_ids=input_ids, attention_mask=attention_mask,\n            token_type_ids=token_type_ids\n        )\n        se

In [5]:
# model definition
'''class MultiTaskSentencePrediction(BertPreTrainedModel):
    def __init__(self, config, num_xpos_labels, num_deprel_labels, num_upos_labels):
        super().__init__(config)
        self.num_xpos_labels = num_xpos_labels
        self.num_deprel_labels = num_deprel_labels
        self.num_upos_labels = num_upos_labels

        self.bert = BertModel(config)

        self.xpos_classifier = nn.Linear(config.hidden_size, num_xpos_labels)
        self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)
        self.upos_classifier = nn.Linear(config.hidden_size, num_upos_labels)

        classifier_dropout = config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        
        self.dropout = nn.Dropout(classifier_dropout)
        self.init_weights()

    def forward(
            self, input_ids, attention_mask=None, token_type_ids=None, 
            xpos_label=None, deprel_label=None, upos_label=None 
    ):
        outputs = self.bert(
            input_ids=input_ids, attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        sequence_output = self.dropout(outputs[0])  # [batch_size, seq_len, hidden_size]


        
        # Classificação por token
        logits_xpos = self.xpos_classifier(sequence_output)     # [batch_size, seq_len, num_xpos_labels]
        logits_deprel = self.deprel_classifier(sequence_output)
        logits_upos = self.upos_classifier(sequence_output)


        loss = None
        if xpos_label != None and deprel_label != None and upos_label != None:
            
            
            loss_fct1 = nn.CrossEntropyLoss(ignore_index=-100)
            loss_fct2 = nn.CrossEntropyLoss(ignore_index=-100)
            loss_fct3 = nn.CrossEntropyLoss(ignore_index=-100)

            loss = loss_fct1(
            logits_xpos.view(-1, self.num_xpos_labels),  # [B * L, num_xpos_labels]
            xpos_label.view(-1)                          # [B * L]
        ) + loss_fct2(
            logits_deprel.view(-1, self.num_deprel_labels),
            deprel_label.view(-1)
        ) + loss_fct3(
            logits_upos.view(-1, self.num_upos_labels),
            upos_label.view(-1)
        )

        return (loss, logits_xpos, logits_deprel, logits_upos) if loss is not None else (logits_xpos, logits_deprel, logits_upos)'''

'class MultiTaskSentencePrediction(BertPreTrainedModel):\n    def __init__(self, config, num_xpos_labels, num_deprel_labels, num_upos_labels):\n        super().__init__(config)\n        self.num_xpos_labels = num_xpos_labels\n        self.num_deprel_labels = num_deprel_labels\n        self.num_upos_labels = num_upos_labels\n\n        self.bert = BertModel(config)\n\n        self.xpos_classifier = nn.Linear(config.hidden_size, num_xpos_labels)\n        self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)\n        self.upos_classifier = nn.Linear(config.hidden_size, num_upos_labels)\n\n        classifier_dropout = config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob\n        \n        self.dropout = nn.Dropout(classifier_dropout)\n        self.init_weights()\n\n    def forward(\n            self, input_ids, attention_mask=None, token_type_ids=None, \n            xpos_label=None, deprel_label=None, upos_label=None \n    ):

In [6]:
# ============================================================
# dependencies.py  (ou célula do notebook)
# ============================================================
import torch
import torch.nn as nn
import numpy as np
from typing import Optional

from transformers import BertModel, BertPreTrainedModel


# ─────────────────────────────────────────────
# 1.  MLP não-linear com inicialização segura
# ─────────────────────────────────────────────
class MLP(nn.Module):
    """Projeção não-linear usada antes das camadas biaffine."""

    def __init__(self, in_features: int, out_features: int, dropout: float = 0.33):
        super().__init__()
        self.linear     = nn.Linear(in_features, out_features)
        self.activation = nn.ELU()
        self.norm       = nn.LayerNorm(out_features)  # normaliza entrada para biaffine
        self.dropout    = nn.Dropout(dropout)

        nn.init.orthogonal_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # LayerNorm garante σ≈1 antes de entrar na biaffine
        return self.dropout(self.norm(self.activation(self.linear(x))))


# ─────────────────────────────────────────────
# 2.  Biaffine  (Dozat & Manning, 2017)
# ─────────────────────────────────────────────
class Biaffine(nn.Module):
    """
    score[b, o, i, j] = Σ_h Σ_k  x[b,i,h] · W[o,h,k] · y[b,j,k]

    Convenção:
      x  → representação de DEPENDENTE   (token que recebe a aresta)
      y  → representação de HEAD         (token que emite a aresta)

    Arc scoring  : out_features=1,           bias_x=True,  bias_y=False
    Rel scoring  : out_features=num_deprel,  bias_x=True,  bias_y=True
    """

    def __init__(
        self,
        in_features:  int,
        out_features: int  = 1,
        bias_x:       bool = True,
        bias_y:       bool = True,
    ):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.bias_x       = bias_x
        self.bias_y       = bias_y

        self.weight = nn.Parameter(
            torch.zeros(
                out_features,
                in_features + int(bias_x),
                in_features + int(bias_y),
            )
        )
        # Escala correta para forma bilinear: Var(s)=H²·σ²_x·σ²_W·σ²_y=1
        # com std=1/H → Var(s)≈1 (evita saturação do softmax desde o step 0)
        nn.init.normal_(self.weight, std=1.0 / in_features)

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """
        x : [B, L, in_features]  — dependente
        y : [B, L, in_features]  — head candidato
        →   [B, out_features, L_dep, L_head]
        """
        if self.bias_x:
            x = torch.cat([x, x.new_ones(*x.shape[:-1], 1)], dim=-1)  # [B, L, H+1]
        if self.bias_y:
            y = torch.cat([y, y.new_ones(*y.shape[:-1], 1)], dim=-1)  # [B, L, H+1]

        # einsum: dep[b,i,h] · W[o,h,k] · head[b,j,k] → [b,o,i,j]
        return torch.einsum("bih,ohk,bjk->boij", x, self.weight, y)

In [7]:
# ─────────────────────────────────────────────
# 3.  Modelo MTL
# ─────────────────────────────────────────────
class MultiTaskSentencePredictionEncoder(BertPreTrainedModel):
    """
    MTL para Universal Dependencies:
      • UPOS    → classificador linear sobre saída do encoder
      • HEAD    → biaffine arc  (out=1)
      • DEPREL  → biaffine rel  (out=num_deprel), avaliado na head predita
    """

    def __init__(
        self,
        config,
        num_deprel_labels: int,
        num_upos_labels:   int,
        arc_hidden:  int   = 500,
        rel_hidden:  int   = 100,
        mlp_dropout: float = 0.33,
    ):
        super().__init__(config)

        self.num_deprel_labels = num_deprel_labels
        self.num_upos_labels   = num_upos_labels
        self.arc_hidden        = arc_hidden
        self.rel_hidden        = rel_hidden

        # Persistir no config para recarregar checkpoints
        config.num_deprel_labels = num_deprel_labels
        config.num_upos_labels   = num_upos_labels
        config.arc_hidden        = arc_hidden
        config.rel_hidden        = rel_hidden

        self.bert = BertModel(config, add_pooling_layer=False)

        encoder_dropout = (
            config.classifier_dropout
            if getattr(config, "classifier_dropout", None) is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(encoder_dropout)

        # ── UPOS: linear direto sobre o encoder ──────────────────────
        self.upos_classifier = nn.Linear(config.hidden_size, num_upos_labels)
        nn.init.xavier_uniform_(self.upos_classifier.weight)
        nn.init.zeros_(self.upos_classifier.bias)

        # ── 4 MLPs: dois para arc, dois para rel ─────────────────────
        self.arc_head_mlp = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.arc_dep_mlp  = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.rel_head_mlp = MLP(config.hidden_size, rel_hidden, mlp_dropout)
        self.rel_dep_mlp  = MLP(config.hidden_size, rel_hidden, mlp_dropout)

        # ── Biaffines ────────────────────────────────────────────────
        # Arc: bias_y=False conforme Dozat & Manning 2017
        self.arc_biaffine = Biaffine(arc_hidden, out_features=1,
                                     bias_x=True, bias_y=False)
        # Rel: ambos os biases
        self.rel_biaffine = Biaffine(rel_hidden, out_features=num_deprel_labels,
                                     bias_x=True, bias_y=True)

        self.post_init()

    def _init_weights(self, module: nn.Module) -> None:
        """
        Estende o _init_weights do BERT para cobrir Biaffine (nn.Parameter direto).
        Chamado por apply() dentro de post_init() — e novamente após from_pretrained
        com _fast_init=False para garantir que nenhum parâmetro fique sem init.
        """
        if isinstance(module, Biaffine):
            # Escala correta para bilinear s = x^T W y: std = 1/H → Var(s) ≈ 1
            nn.init.normal_(module.weight, std=1.0 / module.in_features)
        elif isinstance(module, nn.Linear):
            # Cobre MLP.linear, upos_classifier e todas as camadas BERT
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

    # ── forward ──────────────────────────────────────────────────────
    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        token_type_ids: Optional[torch.Tensor] = None,
        deprel_label:   Optional[torch.Tensor] = None,   # [B, L]
        upos_label:     Optional[torch.Tensor] = None,   # [B, L]
        head_label:     Optional[torch.Tensor] = None,   # [B, L]  valores ∈ {-100, 0..L-1}
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        seq = self.dropout(outputs.last_hidden_state)  # [B, L, H]
        B, L, _ = seq.shape

        # ── UPOS ─────────────────────────────────────────────────────
        logits_upos = self.upos_classifier(seq)          # [B, L, num_upos]

        # ── Arc scores ───────────────────────────────────────────────
        h_arc_dep  = self.arc_dep_mlp(seq)               # [B, L, arc_hidden]
        h_arc_head = self.arc_head_mlp(seq)
        # [B, 1, L_dep, L_head] → [B, L_dep, L_head]
        logits_head = self.arc_biaffine(h_arc_dep, h_arc_head).squeeze(1)

        # Guard: logits > 50 indicam corrupção CUDA ou explosão de gradiente
        # Com CUDA limpo e init correto, max esperado ≈ 6; nunca ultrapassa 50.
        if self.training and logits_head.abs().max() > 1e4:
            raise RuntimeError(
                f"logits_head.abs().max()={logits_head.abs().max():.3e} — "
                "contexto CUDA corrompido. Reinicie o kernel: Kernel → Restart → Run All Cells"
            )

        # Mascarar posições PAD como candidatos a HEAD (-1e4 em vez de -inf:
        # evita NaN em log_softmax quando a linha inteira seria -inf).
        if attention_mask is not None:
            pad_head_mask = (attention_mask == 0).unsqueeze(1)  # [B, 1, L_head]
            logits_head = logits_head.masked_fill(pad_head_mask, -1e4)

        # ── Rel scores ───────────────────────────────────────────────
        h_rel_dep  = self.rel_dep_mlp(seq)               # [B, L, rel_hidden]
        h_rel_head = self.rel_head_mlp(seq)
        # [B, num_deprel, L_dep, L_head]
        logits_rel = self.rel_biaffine(h_rel_dep, h_rel_head)

        # ── Selecionar logits de deprel na head predita ───────────────
        # Usado na saída (compute_metrics) — heads preditas, não gold
        # argmax sobre logits_head já mascarados → nunca retorna posição PAD
        arc_preds = logits_head.argmax(-1).clamp(0, L - 1)         # [B, L]
        idx_pred = (
            arc_preds
            .unsqueeze(-1).unsqueeze(-1)
            .expand(B, L, 1, self.num_deprel_labels)
        )
        logits_rel_t      = logits_rel.permute(0, 2, 3, 1).contiguous()  # [B, L, L, num_deprel]
        logits_deprel_out = logits_rel_t.gather(2, idx_pred).squeeze(2)  # [B, L, num_deprel]

        # ── Loss (teacher forcing com gold heads) ─────────────────────
        loss = None
        if head_label is not None and deprel_label is not None and upos_label is not None:
            loss = self._compute_loss(
                logits_head, logits_rel, logits_upos,
                head_label, deprel_label, upos_label,
                B, L,
            )

        # Formato: (loss, deprel_logits, upos_logits, head_logits)
        # Compatível com o compute_metrics existente
        if loss is not None:
            return (loss, logits_deprel_out, logits_upos, logits_head)
        return (logits_deprel_out, logits_upos, logits_head)

    def _compute_loss(
        self,
        logits_head:  torch.Tensor,   # [B, L, L]
        logits_rel:   torch.Tensor,   # [B, num_deprel, L, L]
        logits_upos:  torch.Tensor,   # [B, L, num_upos]
        head_label:   torch.Tensor,   # [B, L]
        deprel_label: torch.Tensor,   # [B, L]
        upos_label:   torch.Tensor,   # [B, L]
        B: int,
        L: int,
    ) -> torch.Tensor:
        loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

        # ── Sanitização: head OOB causa CUDA device-side assertion ─────────────
        # Frases truncadas podem ter HEAD apontando para posições além de L.
        # CrossEntropyLoss com target >= num_classes → NaN/crash em CUDA.
        oob_mask = (head_label != -100) & (head_label >= L)
        head_label_clean   = head_label.clone()
        deprel_label_clean = deprel_label.clone()
        head_label_clean[oob_mask]   = -100
        deprel_label_clean[oob_mask] = -100  # deprel sem head válido também invalida

        # ── HEAD: cross-entropy sobre L candidatos ────────────────────────────
        loss_head = loss_fct(
            logits_head.reshape(B * L, L),         # [B*L, L]
            head_label_clean.reshape(-1),           # [B*L] ∈ {-100, 0..L-1}
        )

        # ── DEPREL: teacher forcing com gold head ─────────────────────────────
        # Clamp duplo: gather exige índices em [0, L-1].
        # head_label_clean já tem -100 onde é inválido; clampar mapeia -100→0
        # (posição dummy, mas deprel_label_clean=-100 lá → ignorado pelo loss).
        safe_heads = head_label_clean.clamp(0, L - 1)              # [B, L]
        idx_gold = (
            safe_heads
            .unsqueeze(-1).unsqueeze(-1)
            .expand(B, L, 1, self.num_deprel_labels)
        )
        # .contiguous() antes de gather em tensor permutado
        logits_rel_t       = logits_rel.permute(0, 2, 3, 1).contiguous()  # [B, L, L, num_deprel]
        logits_deprel_gold = logits_rel_t.gather(2, idx_gold).squeeze(2)  # [B, L, num_deprel]

        loss_deprel = loss_fct(
            logits_deprel_gold.reshape(B * L, self.num_deprel_labels),
            deprel_label_clean.reshape(-1),
        )

        # ── UPOS ──────────────────────────────────────────────────────────────
        loss_upos = loss_fct(
            logits_upos.reshape(B * L, self.num_upos_labels),
            upos_label.reshape(-1),
        )

        # Detecta perda numericamente inválida antes de retornar
        # (evita propagar NaN/Inf que corrompem pesos silenciosamente)
        for name, val in [("loss_head", loss_head), ("loss_deprel", loss_deprel), ("loss_upos", loss_upos)]:
            if torch.isnan(val) or torch.isinf(val):
                raise RuntimeError(
                    f"{name}={val.item():.4e} — verifique head_label (range: "
                    f"{head_label_clean[head_label_clean != -100].tolist()[:5]}...) "
                    f"e logits (max={logits_head.abs().max().item():.3e})"
                )

        return loss_head + loss_deprel + loss_upos

In [8]:
from transformers import ModernBertModel, ModernBertPreTrainedModel

# ─────────────────────────────────────────────
# 3b.  Modelo MTL — ModernBERT
# ─────────────────────────────────────────────
class MultiTaskSentencePredictionEncoderModern(ModernBertPreTrainedModel):
    """
    MTL para Universal Dependencies (ModernBERT):
      • UPOS    → classificador linear sobre saída do encoder
      • HEAD    → biaffine arc  (out=1)
      • DEPREL  → biaffine rel  (out=num_deprel), avaliado na head predita

    Diferenças em relação à versão BERT:
      - Herda de ModernBertPreTrainedModel (base_model_prefix = "model")
      - Encoder armazenado como self.model para coincidir com as chaves do checkpoint
      - token_type_ids ignorado: ModernBERT não possui token-type embeddings
      - Fallback de dropout compatível com ModernBertConfig
    """

    def __init__(
        self,
        config,
        num_deprel_labels: int,
        num_upos_labels:   int,
        arc_hidden:  int   = 500,
        rel_hidden:  int   = 100,
        mlp_dropout: float = 0.33,
    ):
        super().__init__(config)

        self.num_deprel_labels = num_deprel_labels
        self.num_upos_labels   = num_upos_labels
        self.arc_hidden        = arc_hidden
        self.rel_hidden        = rel_hidden

        # Persistir no config para recarregar checkpoints
        config.num_deprel_labels = num_deprel_labels
        config.num_upos_labels   = num_upos_labels
        config.arc_hidden        = arc_hidden
        config.rel_hidden        = rel_hidden

        # "model" coincide com ModernBertPreTrainedModel.base_model_prefix
        self.model = ModernBertModel(config)

        # ModernBertConfig não tem hidden_dropout_prob; usa embedding_dropout como fallback
        encoder_dropout = (
            getattr(config, "classifier_dropout", None)
            or getattr(config, "hidden_dropout_prob", None)
            or getattr(config, "embedding_dropout", 0.1)
        )
        self.dropout = nn.Dropout(encoder_dropout)

        # ── UPOS: linear direto sobre o encoder ──────────────────────
        self.upos_classifier = nn.Linear(config.hidden_size, num_upos_labels)
        nn.init.xavier_uniform_(self.upos_classifier.weight)
        nn.init.zeros_(self.upos_classifier.bias)

        # ── 4 MLPs: dois para arc, dois para rel ─────────────────────
        self.arc_head_mlp = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.arc_dep_mlp  = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.rel_head_mlp = MLP(config.hidden_size, rel_hidden, mlp_dropout)
        self.rel_dep_mlp  = MLP(config.hidden_size, rel_hidden, mlp_dropout)

        # ── Biaffines ────────────────────────────────────────────────
        self.arc_biaffine = Biaffine(arc_hidden, out_features=1,
                                     bias_x=True, bias_y=False)
        self.rel_biaffine = Biaffine(rel_hidden, out_features=num_deprel_labels,
                                     bias_x=True, bias_y=True)

        self.post_init()

    def _init_weights(self, module: nn.Module) -> None:
        """
        Estende o _init_weights do ModernBERT para cobrir Biaffine (nn.Parameter direto).
        """
        if isinstance(module, Biaffine):
            nn.init.normal_(module.weight, std=1.0 / module.in_features)
        elif isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

    # ── forward ──────────────────────────────────────────────────────
    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        token_type_ids: Optional[torch.Tensor] = None,   # ignorado: ModernBERT não usa
        deprel_label:   Optional[torch.Tensor] = None,   # [B, L]
        upos_label:     Optional[torch.Tensor] = None,   # [B, L]
        head_label:     Optional[torch.Tensor] = None,   # [B, L]  valores ∈ {-100, 0..L-1}
    ):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            # token_type_ids não é suportado pelo ModernBERT
        )
        seq = self.dropout(outputs.last_hidden_state)  # [B, L, H]
        B, L, _ = seq.shape

        # ── UPOS ─────────────────────────────────────────────────────
        logits_upos = self.upos_classifier(seq)          # [B, L, num_upos]

        # ── Arc scores ───────────────────────────────────────────────
        h_arc_dep  = self.arc_dep_mlp(seq)               # [B, L, arc_hidden]
        h_arc_head = self.arc_head_mlp(seq)
        logits_head = self.arc_biaffine(h_arc_dep, h_arc_head).squeeze(1)

        if self.training and logits_head.abs().max() > 1e4:
            raise RuntimeError(
                f"logits_head.abs().max()={logits_head.abs().max():.3e} — "
                "contexto CUDA corrompido. Reinicie o kernel: Kernel → Restart → Run All Cells"
            )

        if attention_mask is not None:
            pad_head_mask = (attention_mask == 0).unsqueeze(1)  # [B, 1, L_head]
            logits_head = logits_head.masked_fill(pad_head_mask, -1e4)

        # ── Rel scores ───────────────────────────────────────────────
        h_rel_dep  = self.rel_dep_mlp(seq)               # [B, L, rel_hidden]
        h_rel_head = self.rel_head_mlp(seq)
        logits_rel = self.rel_biaffine(h_rel_dep, h_rel_head)  # [B, num_deprel, L, L]

        # ── Selecionar logits de deprel na head predita ───────────────
        arc_preds = logits_head.argmax(-1).clamp(0, L - 1)         # [B, L]
        idx_pred = (
            arc_preds
            .unsqueeze(-1).unsqueeze(-1)
            .expand(B, L, 1, self.num_deprel_labels)
        )
        logits_rel_t      = logits_rel.permute(0, 2, 3, 1).contiguous()  # [B, L, L, num_deprel]
        logits_deprel_out = logits_rel_t.gather(2, idx_pred).squeeze(2)  # [B, L, num_deprel]

        # ── Loss (teacher forcing com gold heads) ─────────────────────
        loss = None
        if head_label is not None and deprel_label is not None and upos_label is not None:
            loss = self._compute_loss(
                logits_head, logits_rel, logits_upos,
                head_label, deprel_label, upos_label,
                B, L,
            )

        if loss is not None:
            return (loss, logits_deprel_out, logits_upos, logits_head)
        return (logits_deprel_out, logits_upos, logits_head)

    def _compute_loss(
        self,
        logits_head:  torch.Tensor,
        logits_rel:   torch.Tensor,
        logits_upos:  torch.Tensor,
        head_label:   torch.Tensor,
        deprel_label: torch.Tensor,
        upos_label:   torch.Tensor,
        B: int,
        L: int,
    ) -> torch.Tensor:
        loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

        oob_mask = (head_label != -100) & (head_label >= L)
        head_label_clean   = head_label.clone()
        deprel_label_clean = deprel_label.clone()
        head_label_clean[oob_mask]   = -100
        deprel_label_clean[oob_mask] = -100

        loss_head = loss_fct(
            logits_head.reshape(B * L, L),
            head_label_clean.reshape(-1),
        )

        safe_heads = head_label_clean.clamp(0, L - 1)
        idx_gold = (
            safe_heads
            .unsqueeze(-1).unsqueeze(-1)
            .expand(B, L, 1, self.num_deprel_labels)
        )
        logits_rel_t       = logits_rel.permute(0, 2, 3, 1).contiguous()
        logits_deprel_gold = logits_rel_t.gather(2, idx_gold).squeeze(2)

        loss_deprel = loss_fct(
            logits_deprel_gold.reshape(B * L, self.num_deprel_labels),
            deprel_label_clean.reshape(-1),
        )

        loss_upos = loss_fct(
            logits_upos.reshape(B * L, self.num_upos_labels),
            upos_label.reshape(-1),
        )

        for name, val in [("loss_head", loss_head), ("loss_deprel", loss_deprel), ("loss_upos", loss_upos)]:
            if torch.isnan(val) or torch.isinf(val):
                raise RuntimeError(
                    f"{name}={val.item():.4e} — verifique head_label (range: "
                    f"{head_label_clean[head_label_clean != -100].tolist()[:5]}...) "
                    f"e logits (max={logits_head.abs().max().item():.3e})"
                )

        return loss_head + loss_deprel + loss_upos


In [9]:
# ─────────────────────────────────────────────
# 4.  Fábrica de modelos MTL
# ─────────────────────────────────────────────
def build_model(
    name_model:        str,
    num_deprel_labels: int,
    num_upos_labels:   int,
    arc_hidden:  int   = 500,
    rel_hidden:  int   = 100,
    mlp_dropout: float = 0.33,
):
    """
    Instancia a classe MTL correta de acordo com a arquitetura do modelo pretrained.

    model_type == 'modernbert'  →  MultiTaskSentencePredictionEncoderModern
    qualquer outro (bert, ...)  →  MultiTaskSentencePredictionEncoder
    """
    from transformers import AutoConfig

    config = AutoConfig.from_pretrained(name_model)

    if config.model_type == "modernbert":
        cls = MultiTaskSentencePredictionEncoderModern
    else:
        cls = MultiTaskSentencePredictionEncoder

    print(f"[build_model] model_type='{config.model_type}' → {cls.__name__}")

    return cls.from_pretrained(
        name_model,
        config=config,
        num_deprel_labels=num_deprel_labels,
        num_upos_labels=num_upos_labels,
        arc_hidden=arc_hidden,
        rel_hidden=rel_hidden,
        mlp_dropout=mlp_dropout,
        _fast_init=False,
    ).to("cuda" if torch.cuda.is_available() else "cpu")


In [10]:
from transformers import AutoConfig

# Config sempre carregado do checkpoint: garante arquitetura correta
# (BERT, ModernBERT, mBERT, etc.) independente do modelo base.
MODEL_CONFIG = AutoConfig.from_pretrained(FINETUNED_MODEL_PATH)
TOKENIZER    = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

print(f'Arquitetura: {MODEL_CONFIG.model_type} | hidden_size: {MODEL_CONFIG.hidden_size}')
print(f'Tokenizer  : {TOKENIZER_NAME}')


Arquitetura: modernbert | hidden_size: 768
Tokenizer  : amadeusai/modernJabuticaBERT-Base-1k


In [11]:
# loading model 
model = build_model(
    FINETUNED_MODEL_PATH,
    num_deprel_labels=len(DEPREL_LABELS),
    num_upos_labels=len(UPOS_LABELS)
)

print('Model loaded successfully...')

[build_model] model_type='modernbert' → MultiTaskSentencePredictionEncoderModern


Loading weights: 100%|██████████| 154/154 [00:00<00:00, 11990.62it/s]


Model loaded successfully...


In [12]:
print(len(DEPREL_LABELS))

44


In [13]:
print(len(UPOS_LABELS))

16


In [14]:
from transformers import AutoConfig

In [15]:
#config = AutoConfig.from_pretrained(FINETUNED_MODEL_PATH)

"""model = MultiTaskSentencePredictionEncoder.from_pretrained(
            FINETUNED_MODEL_PATH,
            config=config,
            num_deprel_labels=len(DEPREL_LABELS),
            num_upos_labels=len(UPOS_LABELS)
        ).to("cuda" if torch.cuda.is_available() else "cpu")
"""

'model = MultiTaskSentencePredictionEncoder.from_pretrained(\n            FINETUNED_MODEL_PATH,\n            config=config,\n            num_deprel_labels=len(DEPREL_LABELS),\n            num_upos_labels=len(UPOS_LABELS)\n        ).to("cuda" if torch.cuda.is_available() else "cpu")\n'

In [16]:
'''import pandas as pd

def get_predictions_on_dataframe(sentences, model, tokenizer, device="cpu"):
    predictions_xpos = []
    predictions_deprel = []
    probability_xpos = []
    probability_deprel = []

    model.eval()
    model.to(device)

    for tokens in tqdm(sentences):  # sentences é list[list[str]]
        inputs = tokenizer(
            tokens,
            is_split_into_words=True,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            model_outputs = model(**inputs)

        logits_xpos = model_outputs[0]
        logits_deprel = model_outputs[1]

        word_ids = inputs.word_ids(batch_index=0)

        sent_preds_xpos, sent_preds_deprel = [], []
        sent_probs_xpos, sent_probs_deprel = [], []

        for word_idx in range(len(tokens)):
            subtoken_idxs = [i for i, w_id in enumerate(word_ids) if w_id == word_idx]
            if not subtoken_idxs:
                continue

            first_sub = subtoken_idxs[0]

            prob_xpos = torch.softmax(logits_xpos[0, first_sub], dim=-1)
            prob_deprel = torch.softmax(logits_deprel[0, first_sub], dim=-1)

            pred_xpos = torch.argmax(prob_xpos).item()
            pred_deprel = torch.argmax(prob_deprel).item()

            sent_preds_xpos.append(IDX_TO_XPOS_LABELS[pred_xpos])
            sent_preds_deprel.append(IDX_TO_DEPREL_LABELS[pred_deprel])
            sent_probs_xpos.append(prob_xpos[pred_xpos].item())
            sent_probs_deprel.append(prob_deprel[pred_deprel].item())

        predictions_xpos.append(sent_preds_xpos)
        predictions_deprel.append(sent_preds_deprel)
        probability_xpos.append(sent_probs_xpos)
        probability_deprel.append(sent_probs_deprel)

    # monta DataFrame no final
    df_out = pd.DataFrame({
        "tokens": sentences,
        "xpos_predictions": predictions_xpos,
        "xpos_pred_probability": probability_xpos,
        "deprel_predictions": predictions_deprel,
        "deprel_pred_probability": probability_deprel
    })
    return df_out
'''

'import pandas as pd\n\ndef get_predictions_on_dataframe(sentences, model, tokenizer, device="cpu"):\n    predictions_xpos = []\n    predictions_deprel = []\n    probability_xpos = []\n    probability_deprel = []\n\n    model.eval()\n    model.to(device)\n\n    for tokens in tqdm(sentences):  # sentences é list[list[str]]\n        inputs = tokenizer(\n            tokens,\n            is_split_into_words=True,\n            return_tensors="pt",\n            padding=True,\n            truncation=True\n        ).to(device)\n\n        with torch.no_grad():\n            model_outputs = model(**inputs)\n\n        logits_xpos = model_outputs[0]\n        logits_deprel = model_outputs[1]\n\n        word_ids = inputs.word_ids(batch_index=0)\n\n        sent_preds_xpos, sent_preds_deprel = [], []\n        sent_probs_xpos, sent_probs_deprel = [], []\n\n        for word_idx in range(len(tokens)):\n            subtoken_idxs = [i for i, w_id in enumerate(word_ids) if w_id == word_idx]\n            if n

In [17]:
'''import pandas as pd
import torch
from tqdm import tqdm

def get_predictions_on_dataframe(sentences, model, tokenizer, device="cpu"):
    predictions_xpos = []
    predictions_deprel = []
    predictions_upos = []

    probability_xpos = []
    probability_deprel = []
    probability_upos = []

    model.eval()
    model.to(device)

    for tokens in tqdm(sentences):
        inputs = tokenizer(
            tokens,
            is_split_into_words=True,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            model_outputs = model(**inputs)

        #print(tokens)
        #print(model_outputs[:-1][-1][0, 0].detach().numpy())

        logits_xpos = model_outputs[0]  # cabeça 0
        logits_deprel = model_outputs[1]  # cabeça 1
        logits_upos = model_outputs[2] # cabeça 2

        word_ids = inputs.word_ids(batch_index=0)

        sent_preds_xpos, sent_preds_deprel, sent_preds_upos = [], [], []
        sent_probs_xpos, sent_probs_deprel, sent_probs_upos = [], [], []

        for token_idx in range(len(tokens)):
            # todos os subtokens do token
            subtoken_idxs = [i for i, w_id in enumerate(word_ids) if w_id == token_idx]

            if subtoken_idxs:
                first_sub = subtoken_idxs[0]  # ou média/max dos subtokens
                prob_xpos = torch.softmax(logits_xpos[0, first_sub], dim=-1)
                prob_deprel = torch.softmax(logits_deprel[0, first_sub], dim=-1)
                prob_upos = torch.softmax(logits_upos[0, first_sub], dim=-1)

                pred_xpos = torch.argmax(prob_xpos).item()
                pred_deprel = torch.argmax(prob_deprel).item()
                pred_upos = torch.argmax(prob_upos).item()

                sent_preds_xpos.append(IDX_TO_XPOS_LABELS[pred_xpos])
                sent_preds_deprel.append(IDX_TO_DEPREL_LABELS[pred_deprel])
                sent_preds_upos.append(IDX_TO_UPOS_LABELS[pred_upos])

                sent_probs_xpos.append(prob_xpos[pred_xpos].item())
                sent_probs_deprel.append(prob_deprel[pred_deprel].item())
                sent_probs_upos.append(prob_upos[pred_upos].item())

            else:
                # mantém correspondência de tamanho com token original
                sent_preds_xpos.append(None)
                sent_preds_deprel.append(None)
                sent_preds_upos.append(None)
                
                sent_probs_xpos.append(None)
                sent_probs_deprel.append(None)
                sent_probs_upos.append(None)

        predictions_xpos.append(sent_preds_xpos)
        predictions_deprel.append(sent_preds_deprel)
        predictions_upos.append(sent_preds_upos)

        probability_xpos.append(sent_probs_xpos)
        probability_deprel.append(sent_probs_deprel)
        probability_upos.append(sent_probs_upos)

    # monta DataFrame
    df_out = pd.DataFrame({
        "tokens": sentences,
        "xpos_predictions": predictions_xpos,
        "xpos_pred_probability": probability_xpos,
        "deprel_predictions": predictions_deprel,
        "deprel_pred_probability": probability_deprel,
        "upos_predictions": predictions_upos,
        "upos_pred_probability": probability_upos       
    })

    return df_out'''


'import pandas as pd\nimport torch\nfrom tqdm import tqdm\n\ndef get_predictions_on_dataframe(sentences, model, tokenizer, device="cpu"):\n    predictions_xpos = []\n    predictions_deprel = []\n    predictions_upos = []\n\n    probability_xpos = []\n    probability_deprel = []\n    probability_upos = []\n\n    model.eval()\n    model.to(device)\n\n    for tokens in tqdm(sentences):\n        inputs = tokenizer(\n            tokens,\n            is_split_into_words=True,\n            return_tensors="pt",\n            padding=True,\n            truncation=True\n        ).to(device)\n\n        with torch.no_grad():\n            model_outputs = model(**inputs)\n\n        #print(tokens)\n        #print(model_outputs[:-1][-1][0, 0].detach().numpy())\n\n        logits_xpos = model_outputs[0]  # cabeça 0\n        logits_deprel = model_outputs[1]  # cabeça 1\n        logits_upos = model_outputs[2] # cabeça 2\n\n        word_ids = inputs.word_ids(batch_index=0)\n\n        sent_preds_xpos, sent_p

In [18]:
import pandas as pd
import torch
from tqdm import tqdm

def get_predictions_on_dataframe(sentences, model, tokenizer, device="cpu"):
    #predictions_xpos = []
    predictions_deprel = []
    predictions_upos = []
    predictions_head = []

    #probability_xpos = []
    probability_deprel = []
    probability_upos = []
    probability_head = []

    model.eval()
    model.to(device)

    for tokens in tqdm(sentences):
        inputs = tokenizer(
            tokens,
            is_split_into_words=True,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            model_outputs = model(**inputs)

        #print(tokens)
        #print(model_outputs[:-1][-1][0, 0].detach().numpy())

        #logits_xpos = model_outputs[0]  # cabeça 0
        logits_deprel = model_outputs[0]  # cabeça 1
        logits_upos = model_outputs[1] # cabeça 2
        logits_head = model_outputs[2] # cabeça 3

        word_ids = inputs.word_ids(batch_index=0)

        sent_preds_xpos, sent_preds_deprel, sent_preds_upos, sent_preds_head = [], [], [], []
        sent_probs_xpos, sent_probs_deprel, sent_probs_upos, sent_probs_head = [], [], [], []

        for token_idx in range(len(tokens)):
            # todos os subtokens do token
            subtoken_idxs = [i for i, w_id in enumerate(word_ids) if w_id == token_idx]

            if subtoken_idxs:
                first_sub = subtoken_idxs[0]  # ou média/max dos subtokens
                #prob_xpos = torch.softmax(logits_xpos[0, first_sub], dim=-1)
                prob_deprel = torch.softmax(logits_deprel[0, first_sub], dim=-1)
                prob_upos = torch.softmax(logits_upos[0, first_sub], dim=-1)
                prob_head = torch.softmax(logits_head[0, first_sub], dim=-1)

                #pred_xpos = torch.argmax(prob_xpos).item()
                pred_deprel = torch.argmax(prob_deprel).item()
                pred_upos = torch.argmax(prob_upos).item()
                pred_head = torch.argmax(prob_head).item()

                

                #sent_preds_xpos.append(IDX_TO_XPOS_LABELS[pred_xpos])
                sent_preds_deprel.append(IDX_TO_DEPREL_LABELS[pred_deprel])
                sent_preds_upos.append(IDX_TO_UPOS_LABELS[pred_upos])
                sent_preds_head.append(pred_head)

                #sent_probs_xpos.append(prob_xpos[pred_xpos].item())
                sent_probs_deprel.append(prob_deprel[pred_deprel].item())
                sent_probs_upos.append(prob_upos[pred_upos].item())
                sent_probs_head.append(prob_head)

            else:
                # mantém correspondência de tamanho com token original
                #sent_preds_xpos.append(None)
                sent_preds_deprel.append(None)
                sent_preds_upos.append(None)
                sent_preds_head.append(None)

                #sent_probs_xpos.append(None)
                sent_probs_deprel.append(None)
                sent_probs_upos.append(None)
                sent_probs_head.append(None)

        #predictions_xpos.append(sent_preds_xpos)
        predictions_deprel.append(sent_preds_deprel)
        predictions_upos.append(sent_preds_upos)
        predictions_head.append(sent_preds_head)

        #probability_xpos.append(sent_probs_xpos)
        probability_deprel.append(sent_probs_deprel)
        probability_upos.append(sent_probs_upos)
        probability_head.append(sent_probs_head)

    # monta DataFrame
    df_out = pd.DataFrame({
        "tokens": sentences,
        #"xpos_predictions": predictions_xpos,
        #"xpos_pred_probability": probability_xpos,
        "deprel_predictions": predictions_deprel,
        "deprel_pred_probability": probability_deprel,
        "upos_predictions": predictions_upos,
        "upos_pred_probability": probability_upos,
        "head_predictions": predictions_head,
        "head_pred_probability": probability_head    
    })

    return df_out


In [19]:
from datasets import load_from_disk

dataset = load_from_disk('/home/guilhermelima/msc/data_dois/complaints_dataset_obj_outxpos')
test_dataset = dataset['test']
print(f"Test set: {len(test_dataset)} sentenças")

Test set: 1683 sentenças


In [20]:
test_sentences = test_dataset['tokens']
test_upos      = test_dataset['upos']
test_deprel    = test_dataset['deprel']
test_head      = test_dataset['head_tags']

print(f"Sentenças: {len(test_sentences)}")
print(f"Exemplo tokens : {test_sentences[0]}")
print(f"Exemplo upos   : {test_upos[0]}")
print(f"Exemplo deprel : {test_deprel[0]}")
print(f"Exemplo heads  : {test_head[0]}")

Sentenças: 1683
Exemplo tokens : ['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.']
Exemplo upos   : ['DET', 'PROPN', 'PROPN', 'ADV', 'VERB', 'DET', 'NOUN', 'PUNCT']
Exemplo deprel : ['det', 'nsubj', 'flat:name', 'advmod', 'root', 'det', 'obj', 'punct']
Exemplo heads  : [2, 5, 2, 5, 0, 7, 5, 5]


In [21]:
# Células de geração de .conll ignoradas — dados carregados diretamente do dataset de treino
# (test.csv usava anotações UD v1 incompatíveis com o vocabulário do modelo)


In [22]:
#print(data)

In [23]:
#def aling_word_for_sentence(df):

#    word_vector = []

#    for taggings in df:
#        word_vector.append(" ".join(taggings))
#    return word_vector

#test_df['sentence'] = aling_word_for_sentence(test_df['tokens'])

In [24]:
#!pip install conllu

In [25]:
# test_sentences, test_upos, test_deprel, test_head já definidos na célula anterior
print(f"Total de sentenças de teste: {len(test_sentences)}")
print(f"Primeira sentença: {test_sentences[0]}")

Total de sentenças de teste: 1683
Primeira sentença: ['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.']


In [26]:
test_sentences

Column([['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.'], ['A', 'Odebrecht', 'pagou', '300', '%', 'a', 'mais', 'por', 'o', 'direito', 'de', 'explorar', 'o', 'aeroporto', 'de', 'o', 'Galeão', '.'], ['Em', 'o', 'começo', 'de', 'o', 'século', ',', 'a', 'JBS/Friboi', 'chegava', 'a', 'o', 'grupo', 'de', 'as', '400', 'maiores', '.'], ['Os', 'sons', 'indesejáveis', 'emitidos', 'por', 'uma', 'porta', ',', 'por', 'exemplo', ',', 'são', 'eliminados', 'por', 'R$', '150', '.'], ['Que', 'foi', 'herança', 'de', 'o', 'PT', ',', 'que', 'nos', 'deixou', 'esse', 'rombo', ',', 'disse', 'Doria', '.'], ...])

In [27]:
model

MultiTaskSentencePredictionEncoderModern(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
  

In [28]:
predict_test_df = get_predictions_on_dataframe(test_sentences, model, TOKENIZER)

100%|██████████| 1683/1683 [01:10<00:00, 23.98it/s]


In [29]:
predict_test_df

,tokens,deprel_predictions,deprel_pred_probability,upos_predictions,upos_pred_probability,head_predictions,head_pred_probability
0,"[O, Capitão, América, também, bajulou, o, tuca...","[det, nsubj, flat:name, advmod, root, det, obj...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[DET, PROPN, PROPN, ADV, VERB, DET, NOUN, PUNCT]","[0.9999998807907104, 0.9999997615814209, 0.999...","[2, 5, 2, 5, 0, 7, 5, 5]","[[tensor(4.6454e-25), tensor(3.1367e-20), tens..."
1,"[A, Odebrecht, pagou, 300, %, a, mais, por, o,...","[det, nsubj, root, nummod, obj, case, advmod, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 0.9999635219573975, ...","[DET, PROPN, VERB, NUM, SYM, ADP, ADV, ADP, DE...","[0.9999982118606567, 0.9999996423721313, 0.999...","[2, 3, 0, 5, 3, 7, 5, 10, 10, 3, 12, 10, 14, 1...","[[tensor(2.4642e-19), tensor(3.4784e-13), tens..."
2,"[Em, o, começo, de, o, século, ,, a, JBS/Fribo...","[case, det, obl, case, det, nmod, punct, det, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[ADP, DET, NOUN, ADP, DET, NOUN, PUNCT, DET, P...","[0.9999983310699463, 1.0, 0.9999986886978149, ...","[3, 3, 12, 6, 6, 3, 3, 9, 12, 0, 15, 15, 12, 1...","[[tensor(3.2632e-27), tensor(5.6445e-23), tens..."
3,"[Os, sons, indesejáveis, emitidos, por, uma, p...","[det, nsubj:pass, amod, acl, case, det, obl:ag...","[1.0, 0.9999946355819702, 1.0, 1.0, 0.99999988...","[DET, NOUN, ADJ, VERB, ADP, DET, NOUN, PUNCT, ...","[0.9999991655349731, 0.9999986886978149, 0.999...","[2, 13, 2, 2, 7, 7, 4, 10, 10, 13, 10, 13, 0, ...","[[tensor(8.0781e-28), tensor(2.9534e-25), tens..."
4,"[Que, foi, herança, de, o, PT, ,, que, nos, de...","[nsubj, cop, ccomp, case, det, nmod, punct, ns...","[0.9999856948852539, 0.9999969005584717, 0.999...","[PRON, AUX, NOUN, ADP, DET, PROPN, PUNCT, PRON...","[0.9999094009399414, 0.9999698400497437, 0.999...","[3, 3, 14, 6, 6, 3, 10, 10, 10, 6, 12, 10, 10,...","[[tensor(1.1251e-21), tensor(2.8675e-12), tens..."
...,...,...,...,...,...,...,...
1678,"[Julia, Louis-Dreyfus, ganhou, por, a, sexta, ...","[nsubj, flat:name, root, case, det, amod, obl,...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.99999988...","[PROPN, PROPN, VERB, ADP, DET, ADJ, NOUN, ADJ,...","[1.0, 0.9999998807907104, 0.9999752044677734, ...","[3, 1, 0, 7, 7, 7, 3, 7, 10, 3, 13, 13, 3, 16,...","[[tensor(9.2212e-19), tensor(1.7450e-25), tens..."
1679,"[Até, a, família, julga, mais, e, apoia, menos...","[advmod, det, nsubj, ccomp:speech, advmod, cc,...","[0.9999938011169434, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[ADV, DET, NOUN, VERB, ADV, CCONJ, VERB, ADV, ...","[0.9999312162399292, 0.9999998807907104, 0.999...","[3, 3, 4, 15, 4, 7, 4, 7, 13, 11, 13, 13, 7, 4...","[[tensor(2.4573e-19), tensor(1.8739e-14), tens..."
1680,"[Mas, há, episódios, que, indicam, em, Damião,...","[cc, root, obj, nsubj, acl:relcl, case, obl, d...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.999...","[CCONJ, VERB, NOUN, PRON, VERB, ADP, PROPN, DE...","[0.9999806880950928, 0.9999988079071045, 0.999...","[2, 0, 2, 5, 3, 7, 5, 9, 5, 9, 2]","[[tensor(3.7973e-20), tensor(9.7216e-18), tens..."
1681,"["", Mas, Che, permanece, puro, ,, de, certo, m...","[punct, cc, nsubj, ccomp:speech, xcomp, punct,...","[1.0, 1.0, 1.0, 1.0, 0.9999990463256836, 1.0, ...","[PUNCT, CCONJ, PROPN, VERB, ADJ, PUNCT, ADP, A...","[0.9999964237213135, 0.999997615814209, 0.9999...","[4, 4, 4, 13, 4, 9, 9, 9, 4, 4, 4, 13, 0, 13]","[[tensor(1.2638e-24), tensor(9.8979e-24), tens..."


In [30]:
predict_test_df

,tokens,deprel_predictions,deprel_pred_probability,upos_predictions,upos_pred_probability,head_predictions,head_pred_probability
0,"[O, Capitão, América, também, bajulou, o, tuca...","[det, nsubj, flat:name, advmod, root, det, obj...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[DET, PROPN, PROPN, ADV, VERB, DET, NOUN, PUNCT]","[0.9999998807907104, 0.9999997615814209, 0.999...","[2, 5, 2, 5, 0, 7, 5, 5]","[[tensor(4.6454e-25), tensor(3.1367e-20), tens..."
1,"[A, Odebrecht, pagou, 300, %, a, mais, por, o,...","[det, nsubj, root, nummod, obj, case, advmod, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 0.9999635219573975, ...","[DET, PROPN, VERB, NUM, SYM, ADP, ADV, ADP, DE...","[0.9999982118606567, 0.9999996423721313, 0.999...","[2, 3, 0, 5, 3, 7, 5, 10, 10, 3, 12, 10, 14, 1...","[[tensor(2.4642e-19), tensor(3.4784e-13), tens..."
2,"[Em, o, começo, de, o, século, ,, a, JBS/Fribo...","[case, det, obl, case, det, nmod, punct, det, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[ADP, DET, NOUN, ADP, DET, NOUN, PUNCT, DET, P...","[0.9999983310699463, 1.0, 0.9999986886978149, ...","[3, 3, 12, 6, 6, 3, 3, 9, 12, 0, 15, 15, 12, 1...","[[tensor(3.2632e-27), tensor(5.6445e-23), tens..."
3,"[Os, sons, indesejáveis, emitidos, por, uma, p...","[det, nsubj:pass, amod, acl, case, det, obl:ag...","[1.0, 0.9999946355819702, 1.0, 1.0, 0.99999988...","[DET, NOUN, ADJ, VERB, ADP, DET, NOUN, PUNCT, ...","[0.9999991655349731, 0.9999986886978149, 0.999...","[2, 13, 2, 2, 7, 7, 4, 10, 10, 13, 10, 13, 0, ...","[[tensor(8.0781e-28), tensor(2.9534e-25), tens..."
4,"[Que, foi, herança, de, o, PT, ,, que, nos, de...","[nsubj, cop, ccomp, case, det, nmod, punct, ns...","[0.9999856948852539, 0.9999969005584717, 0.999...","[PRON, AUX, NOUN, ADP, DET, PROPN, PUNCT, PRON...","[0.9999094009399414, 0.9999698400497437, 0.999...","[3, 3, 14, 6, 6, 3, 10, 10, 10, 6, 12, 10, 10,...","[[tensor(1.1251e-21), tensor(2.8675e-12), tens..."
...,...,...,...,...,...,...,...
1678,"[Julia, Louis-Dreyfus, ganhou, por, a, sexta, ...","[nsubj, flat:name, root, case, det, amod, obl,...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.99999988...","[PROPN, PROPN, VERB, ADP, DET, ADJ, NOUN, ADJ,...","[1.0, 0.9999998807907104, 0.9999752044677734, ...","[3, 1, 0, 7, 7, 7, 3, 7, 10, 3, 13, 13, 3, 16,...","[[tensor(9.2212e-19), tensor(1.7450e-25), tens..."
1679,"[Até, a, família, julga, mais, e, apoia, menos...","[advmod, det, nsubj, ccomp:speech, advmod, cc,...","[0.9999938011169434, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[ADV, DET, NOUN, VERB, ADV, CCONJ, VERB, ADV, ...","[0.9999312162399292, 0.9999998807907104, 0.999...","[3, 3, 4, 15, 4, 7, 4, 7, 13, 11, 13, 13, 7, 4...","[[tensor(2.4573e-19), tensor(1.8739e-14), tens..."
1680,"[Mas, há, episódios, que, indicam, em, Damião,...","[cc, root, obj, nsubj, acl:relcl, case, obl, d...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.999...","[CCONJ, VERB, NOUN, PRON, VERB, ADP, PROPN, DE...","[0.9999806880950928, 0.9999988079071045, 0.999...","[2, 0, 2, 5, 3, 7, 5, 9, 5, 9, 2]","[[tensor(3.7973e-20), tensor(9.7216e-18), tens..."
1681,"["", Mas, Che, permanece, puro, ,, de, certo, m...","[punct, cc, nsubj, ccomp:speech, xcomp, punct,...","[1.0, 1.0, 1.0, 1.0, 0.9999990463256836, 1.0, ...","[PUNCT, CCONJ, PROPN, VERB, ADJ, PUNCT, ADP, A...","[0.9999964237213135, 0.999997615814209, 0.9999...","[4, 4, 4, 13, 4, 9, 9, 9, 4, 4, 4, 13, 0, 13]","[[tensor(1.2638e-24), tensor(9.8979e-24), tens..."


In [31]:
predict_test_df.to_csv('./predict_test_df_jabuticabert_biaffine.csv', index=False)

In [32]:
for i in range(1):
    print(
        f"Exemplo {i} - Tokens: {len(predict_test_df['tokens'][i])} | "
        #f"XPOS: {len(predict_test_df['xpos_predictions'][i])} | "
        f"DEPREL: {len(predict_test_df['deprel_predictions'][i])}"
    )


Exemplo 0 - Tokens: 8 | DEPREL: 8


In [33]:
print(test_sentences[0])

['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.']


In [34]:
def compute_dependency_metrics(test_sentences, test_upos, test_deprel, test_head, predict_df):
    """
    Métricas padrão para análise de dependências (dissertação):
      UPOS Accuracy — % de tokens com POS tag correta
      UAS           — Unlabeled Attachment Score: HEAD correto
      LAS           — Labeled Attachment Score:   HEAD + DEPREL corretos
    """
    total        = 0
    upos_correct = 0
    uas_correct  = 0
    las_correct  = 0
    skipped      = 0

    for i in range(len(test_sentences)):
        gold_upos   = test_upos[i]
        gold_deprel = test_deprel[i]
        gold_head   = test_head[i]

        pred_upos   = predict_df['upos_predictions'].iloc[i]
        pred_deprel = predict_df['deprel_predictions'].iloc[i]
        pred_head   = predict_df['head_predictions'].iloc[i]

        for j in range(len(gold_head)):
            if pred_head[j] is None or pred_upos[j] is None or pred_deprel[j] is None:
                skipped += 1
                continue

            total += 1

            if pred_upos[j] == gold_upos[j]:
                upos_correct += 1

            # UAS: HEAD correto
            if pred_head[j] == gold_head[j]:
                uas_correct += 1
                # LAS: HEAD correto E DEPREL correto
                if pred_deprel[j] == gold_deprel[j]:
                    las_correct += 1

    return {
        'upos_accuracy': upos_correct / total if total > 0 else 0,
        'uas':           uas_correct  / total if total > 0 else 0,
        'las':           las_correct  / total if total > 0 else 0,
        'total_tokens':  total,
        'skipped':       skipped,
    }


In [35]:
metrics = compute_dependency_metrics(
    test_sentences, test_upos, test_deprel, test_head, predict_test_df
)

print(f"UPOS Accuracy : {metrics['upos_accuracy']:.4f}")
print(f"UAS           : {metrics['uas']:.4f}")
print(f"LAS           : {metrics['las']:.4f}")
print(f"Total tokens  : {metrics['total_tokens']}")
print(f"Ignorados     : {metrics['skipped']}")


UPOS Accuracy : 0.9892
UAS           : 0.8894
LAS           : 0.8744
Total tokens  : 33580
Ignorados     : 0


In [43]:
model

MultiTaskSentencePredictionEncoderModern(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
  

In [36]:
#text = [["O", "gato", "preto", "dorme", "no", "sofá", "."]]
#text = [["A", "menina", "brinca", "no", "parque", "."]]
#text = [["Se", "chover", ",", "o", "jogo", "será", "cancelado", "."]]
text = [['Mas', 'por', 'não', 'existir', 'um', 'marco', 'legal', 'há', 'uma', 'insegurança', 'por', 'parte', 'dos', 'investidores', '"', ',', 'destacou', '.']]

In [37]:
#text = [[".", ".", ".", ".", "", ""]]

#Token = Token_alvo Indice
#Optuna

In [38]:
retorno = get_predictions_on_dataframe(text, model, TOKENIZER)
#aux = [test_df['tokens'][0]]


100%|██████████| 1/1 [00:00<00:00, 20.84it/s]


In [39]:
#retorno = get_predictions_on_dataframe(aux, model, TOKENIZER)

In [40]:
retorno

,tokens,deprel_predictions,deprel_pred_probability,upos_predictions,upos_pred_probability,head_predictions,head_pred_probability
0,"[Mas, por, não, existir, um, marco, legal, há,...","[cc, mark, advmod, advcl, det, nsubj, amod, cc...","[1.0, 1.0, 1.0, 1.0, 1.0, 0.9999979734420776, ...","[CCONJ, ADP, ADV, VERB, DET, NOUN, ADJ, VERB, ...","[0.9999535083770752, 0.9937589168548584, 0.999...","[8, 4, 4, 8, 6, 4, 6, 17, 10, 8, 12, 10, 14, 1...","[[tensor(1.4869e-16), tensor(1.9926e-14), tens..."


In [41]:
retorno

,tokens,deprel_predictions,deprel_pred_probability,upos_predictions,upos_pred_probability,head_predictions,head_pred_probability
0,"[Mas, por, não, existir, um, marco, legal, há,...","[cc, mark, advmod, advcl, det, nsubj, amod, cc...","[1.0, 1.0, 1.0, 1.0, 1.0, 0.9999979734420776, ...","[CCONJ, ADP, ADV, VERB, DET, NOUN, ADJ, VERB, ...","[0.9999535083770752, 0.9937589168548584, 0.999...","[8, 4, 4, 8, 6, 4, 6, 17, 10, 8, 12, 10, 14, 1...","[[tensor(1.4869e-16), tensor(1.9926e-14), tens..."


In [42]:
print('Gold deprel sentença 2:', test_deprel[2])
print('Pred deprel sentença 2:', predict_test_df['deprel_predictions'].iloc[2])

Gold deprel sentença 2: ['case', 'det', 'obl', 'case', 'det', 'nmod', 'punct', 'det', 'nsubj', 'root', 'case', 'det', 'obl', 'case', 'det', 'nmod', 'amod', 'punct']
Pred deprel sentença 2: ['case', 'det', 'obl', 'case', 'det', 'nmod', 'punct', 'det', 'nsubj', 'root', 'case', 'det', 'obl', 'case', 'det', 'nmod', 'amod', 'punct']


# Análise por camada (Logit Lens) — arquitetura Biaffine

Mesma análise do notebook linear, adaptada à arquitetura biaffine (Dozat & Manning, 2017):

- **UPOS**   → classificador linear aplicado à saída de cada camada;
- **HEAD**   → `arc_dep_mlp`/`arc_head_mlp` + `arc_biaffine` aplicados à saída de cada camada
  (scores `[L_dep, L_head]`; predição = argmax sobre candidatos a head);
- **DEPREL** → `rel_dep_mlp`/`rel_head_mlp` + `rel_biaffine`, avaliado na **head predita pela
  própria camada** (mesmo protocolo do forward do modelo).

Camada `0` = embeddings; camadas `1..12` = blocos do encoder.

> ⚠️ As cabeças (MLPs + biaffine) foram treinadas sobre a camada 12 — leituras de camadas
> intermediárias são um limite inferior (logit lens). Modelo: **`biaffine_BERTImbau_base`**.

In [ ]:
from transformers import AutoConfig, AutoTokenizer

# ── Modelo para análise por camada: biaffine_BERTImbau_base ────────────────────
LAYERWISE_MODEL_PATH     = '/home/guilhermelima/msc/biaffine_BERTImbau_base/checkpoint-24321'
LAYERWISE_TOKENIZER_NAME = 'neuralmind/bert-base-portuguese-cased'  # BERTimbau-base

layerwise_tokenizer = AutoTokenizer.from_pretrained(LAYERWISE_TOKENIZER_NAME)
layerwise_model = build_model(
    LAYERWISE_MODEL_PATH,
    num_deprel_labels=len(DEPREL_LABELS),
    num_upos_labels=len(UPOS_LABELS),
)
layerwise_model.eval()

NUM_HIDDEN_LAYERS = layerwise_model.config.num_hidden_layers
print(f'Camadas: {NUM_HIDDEN_LAYERS} | hidden: {layerwise_model.config.hidden_size} | '
      f'arc_hidden: {layerwise_model.arc_hidden} | rel_hidden: {layerwise_model.rel_hidden}')


In [ ]:
def biaffine_heads_from_hidden(model, hs):
    """
    Aplica as cabeças treinadas (upos linear + arc/rel biaffine) sobre estados ocultos.

    hs: [N, L, hidden] — N camadas tratadas como batch.
    Retorna:
      logits_upos [N, L, n_upos] | logits_head [N, L_dep, L_head]
      logits_deprel [N, L, n_deprel] (rel avaliado na head predita por camada)
    """
    logits_upos = model.upos_classifier(hs)

    h_arc_dep, h_arc_head = model.arc_dep_mlp(hs), model.arc_head_mlp(hs)
    logits_head = model.arc_biaffine(h_arc_dep, h_arc_head).squeeze(1)      # [N, L, L]

    h_rel_dep, h_rel_head = model.rel_dep_mlp(hs), model.rel_head_mlp(hs)
    logits_rel = model.rel_biaffine(h_rel_dep, h_rel_head)                  # [N, n_deprel, L, L]

    # deprel na head predita pela própria camada (mesmo protocolo do forward)
    N, L, _ = logits_head.shape
    arc_preds = logits_head.argmax(-1).clamp(0, L - 1)                      # [N, L]
    idx = arc_preds.unsqueeze(-1).unsqueeze(-1).expand(N, L, 1, model.num_deprel_labels)
    logits_rel_t  = logits_rel.permute(0, 2, 3, 1).contiguous()             # [N, L, L, n_deprel]
    logits_deprel = logits_rel_t.gather(2, idx).squeeze(2)                  # [N, L, n_deprel]

    return logits_upos, logits_head, logits_deprel


def get_layerwise_predictions(tokens, model, tokenizer,
                              gold_upos=None, gold_deprel=None, gold_head=None,
                              device=None):
    """
    Logit lens biaffine: DataFrame longo com uma linha por (token, camada, tarefa).
    Mesmo schema do notebook linear: pred, pred_prob, gold, gold_prob, correct.
    """
    if device is None:
        device = next(model.parameters()).device
    model.eval()

    inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                       padding=True, truncation=True).to(device)
    with torch.no_grad():
        bert_out = model.bert(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            token_type_ids=inputs.get('token_type_ids'),
            output_hidden_states=True,
        )
        hs_all = torch.stack(bert_out.hidden_states, dim=0)[:, 0]           # [13, L, H]
        logits_upos, logits_head, logits_deprel = biaffine_heads_from_hidden(model, hs_all)

    word_ids = inputs.word_ids(batch_index=0)
    L = hs_all.shape[1]
    rows = []
    for token_idx, token in enumerate(tokens):
        subtoken_idxs = [i for i, w in enumerate(word_ids) if w == token_idx]
        if not subtoken_idxs:
            continue
        first_sub = subtoken_idxs[0]

        for layer_idx in range(hs_all.shape[0]):
            specs = [
                ('upos',   logits_upos[layer_idx, first_sub],   IDX_TO_UPOS_LABELS,
                 UPOS_LABELS_TO_IDX,   gold_upos),
                ('deprel', logits_deprel[layer_idx, first_sub], IDX_TO_DEPREL_LABELS,
                 DEPREL_LABELS_TO_IDX, gold_deprel),
                ('head',   logits_head[layer_idx, first_sub],   None, None, gold_head),
            ]
            for task, logit_vec, id2label, label2id, gold_seq in specs:
                probs = torch.softmax(logit_vec, dim=-1)
                pred_id = torch.argmax(probs).item()
                pred = id2label[pred_id] if id2label is not None else pred_id

                gold, gold_prob, correct = None, None, None
                if gold_seq is not None:
                    gold = gold_seq[token_idx]
                    gold_id = label2id[gold] if label2id is not None else int(gold)
                    if gold_id < len(probs):
                        gold_prob = probs[gold_id].item()
                        correct = (pred_id == gold_id)
                    else:  # head gold além do comprimento tokenizado (truncamento)
                        gold_prob, correct = 0.0, False

                rows.append({
                    'token_idx': token_idx, 'token': token,
                    'layer': layer_idx, 'task': task,
                    'pred': pred, 'pred_prob': probs[pred_id].item(),
                    'gold': gold, 'gold_prob': gold_prob, 'correct': correct,
                })
    return pd.DataFrame(rows)


In [ ]:
def inspect_token_across_layers(df_layers, token_idx, task='upos'):
    """Tabela camada a camada: como o modelo classifica UM token para UMA tarefa."""
    sel = (df_layers[(df_layers.token_idx == token_idx) & (df_layers.task == task)]
           .sort_values('layer').reset_index(drop=True))
    token, gold = sel['token'].iloc[0], sel['gold'].iloc[0]
    print(f"Token: '{token}' (idx {token_idx}) | tarefa: {task} | gold: {gold}")
    tabela = sel[['layer', 'pred', 'pred_prob', 'gold_prob', 'correct']].copy()
    tabela[['pred_prob', 'gold_prob']] = tabela[['pred_prob', 'gold_prob']].round(4)
    return tabela


def layer_influence(df_layers, token_idx, task='upos'):
    """Δ P(gold) entre camadas consecutivas: qual camada empurrou na direção correta."""
    sel = (df_layers[(df_layers.token_idx == token_idx) & (df_layers.task == task)]
           .sort_values('layer').reset_index(drop=True))
    token, gold = sel['token'].iloc[0], sel['gold'].iloc[0]
    gp   = sel['gold_prob'].to_numpy(dtype=float)
    corr = sel['correct'].to_numpy(dtype=bool)
    deltas = np.diff(gp)

    decision_layer = next((l for l in range(len(corr)) if corr[l:].all()), None)
    best_layer = int(np.argmax(deltas)) + 1

    print(f"Token: '{token}' | tarefa: {task} | gold: {gold}")
    print(f"P(gold) embeddings: {gp[0]:.4f} | P(gold) camada final: {gp[-1]:.4f}")
    print(f"Camada de MAIOR contribuição positiva: {best_layer} (Δ = +{deltas[best_layer-1]:.4f})")
    if decision_layer is not None:
        print(f"Camada de decisão: {decision_layer}")
    else:
        print("Predição final INCORRETA.")

    return pd.DataFrame({
        'layer': np.arange(1, len(gp)),
        'delta_gold_prob': np.round(deltas, 4),
        'gold_prob_acumulada': np.round(gp[1:], 4),
        'pred_da_camada': sel['pred'].iloc[1:].values,
    })


In [ ]:
# ── Exemplo: token 'O' com gold 'DET' (mesmo exemplo do notebook linear) ───────
exemplo_sent_idx, exemplo_tok_idx = None, None
for i, (toks, upos_seq) in enumerate(zip(test_sentences, test_upos)):
    for j, (t, u) in enumerate(zip(toks, upos_seq)):
        if t == 'O' and u == 'DET':
            exemplo_sent_idx, exemplo_tok_idx = i, j
            break
    if exemplo_sent_idx is not None:
        break

print(f"Sentença {exemplo_sent_idx}: {test_sentences[exemplo_sent_idx]}")

df_layers_exemplo = get_layerwise_predictions(
    test_sentences[exemplo_sent_idx], layerwise_model, layerwise_tokenizer,
    gold_upos=test_upos[exemplo_sent_idx],
    gold_deprel=test_deprel[exemplo_sent_idx],
    gold_head=test_head[exemplo_sent_idx],
)
inspect_token_across_layers(df_layers_exemplo, exemplo_tok_idx, task='upos')


In [ ]:
layer_influence(df_layers_exemplo, exemplo_tok_idx, task='upos')


## Impacto por camada em TODO o test set — por tag e por tarefa (biaffine)

Mesmas métricas do notebook linear: `acc` e `mean_gold_prob` por (tarefa, tag, camada),
`best_layer_prob` (camada que mais empurrou na direção correta) e `conv_layer`
(primeira camada com acc ≥ 95% da final). CSVs com sufixo `_biaffine`.

In [ ]:
def layerwise_per_label_stats(sentences, gold_upos_list, gold_deprel_list, gold_head_list,
                              model, tokenizer, device=None, max_sentences=None):
    """Estatísticas por (tarefa, tag gold, camada) — versão biaffine."""
    if device is None:
        device = next(model.parameters()).device
    model.eval()

    n_layers = model.config.num_hidden_layers + 1
    n_lab = {'upos': len(UPOS_LABELS), 'deprel': len(DEPREL_LABELS),
             'head': max(max(h) for h in gold_head_list) + 1}
    correct       = {t: np.zeros((n, n_layers)) for t, n in n_lab.items()}
    gold_prob_sum = {t: np.zeros((n, n_layers)) for t, n in n_lab.items()}
    support       = {t: np.zeros(n)             for t, n in n_lab.items()}

    n_sents = len(sentences) if max_sentences is None else min(max_sentences, len(sentences))
    for i in tqdm(range(n_sents)):
        tokens = sentences[i]
        golds = {
            'upos':   [UPOS_LABELS_TO_IDX[u]   for u in gold_upos_list[i]],
            'deprel': [DEPREL_LABELS_TO_IDX[d] for d in gold_deprel_list[i]],
            'head':   [int(h)                  for h in gold_head_list[i]],
        }
        inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                           padding=True, truncation=True).to(device)
        with torch.no_grad():
            bert_out = model.bert(
                input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'],
                token_type_ids=inputs.get('token_type_ids'), output_hidden_states=True,
            )
            hs_all = torch.stack(bert_out.hidden_states, dim=0)[:, 0]
            logits_upos, logits_head, logits_deprel = biaffine_heads_from_hidden(model, hs_all)

            word_ids = inputs.word_ids(batch_index=0)
            first_subs, tok_idxs = [], []
            seen = set()
            for pos, w in enumerate(word_ids):
                if w is not None and w not in seen and w < len(tokens):
                    seen.add(w)
                    first_subs.append(pos)
                    tok_idxs.append(w)
            if not first_subs:
                continue
            fs = torch.tensor(first_subs, device=device)
            L = hs_all.shape[1]

            all_logits = {'upos': logits_upos[:, fs], 'deprel': logits_deprel[:, fs],
                          'head': logits_head[:, fs]}
            for task in ('upos', 'deprel', 'head'):
                gold_ids_np = np.array([golds[task][t] for t in tok_idxs])
                probs = torch.softmax(all_logits[task], dim=-1)              # [13, n_tok, C]
                preds = probs.argmax(dim=-1).cpu().numpy()                   # [13, n_tok]

                C = probs.shape[-1]
                valid = gold_ids_np < C          # head gold pode exceder L (truncamento)
                safe_ids = np.where(valid, gold_ids_np, 0)
                gp = probs.gather(-1, torch.tensor(safe_ids, device=device)
                                  .expand(len(probs), -1).unsqueeze(-1)).squeeze(-1).cpu().numpy()
                gp[:, ~valid] = 0.0
                corr = (preds == gold_ids_np[None, :]) & valid[None, :]

                for layer_idx in range(len(probs)):
                    np.add.at(correct[task][:, layer_idx],       gold_ids_np, corr[layer_idx])
                    np.add.at(gold_prob_sum[task][:, layer_idx], gold_ids_np, gp[layer_idx])
                np.add.at(support[task], gold_ids_np, 1)

    id2label = {'upos': IDX_TO_UPOS_LABELS, 'deprel': IDX_TO_DEPREL_LABELS, 'head': None}
    stats = {}
    for task, n in n_lab.items():
        rows = []
        for lab_id in range(n):
            sup = support[task][lab_id]
            if sup == 0:
                continue
            label = id2label[task][lab_id] if id2label[task] is not None else lab_id
            for layer in range(n_layers):
                rows.append({'label': label, 'layer': layer,
                             'acc':            correct[task][lab_id, layer] / sup,
                             'mean_gold_prob': gold_prob_sum[task][lab_id, layer] / sup,
                             'support': int(sup)})
        stats[task] = pd.DataFrame(rows)
    return stats


perlabel_stats = layerwise_per_label_stats(
    test_sentences, test_upos, test_deprel, test_head,
    layerwise_model, layerwise_tokenizer,
)
for task, df in perlabel_stats.items():
    df.to_csv(f'./layerwise_per_label_stats_{task}_biaffine.csv', index=False)
    print(f'{task}: {df.label.nunique()} tags | salvo em layerwise_per_label_stats_{task}_biaffine.csv')


In [ ]:
def summarize_layer_impact(stats_df, min_support=30):
    """Camadas mais impactantes por tag (mesma definição do notebook linear)."""
    rows = []
    for label, g in stats_df.groupby('label', sort=False):
        g = g.sort_values('layer')
        sup = g['support'].iloc[0]
        if sup < min_support:
            continue
        acc = g['acc'].to_numpy()
        gp  = g['mean_gold_prob'].to_numpy()
        d_acc, d_gp = np.diff(acc), np.diff(gp)
        final_acc = acc[-1]
        conv = next((l for l in range(len(acc)) if final_acc > 0 and acc[l] >= 0.95 * final_acc), None)
        rows.append({'label': label, 'support': sup,
                     'acc_emb': round(acc[0], 4), 'acc_final': round(final_acc, 4),
                     'best_layer_acc':  int(np.argmax(d_acc)) + 1, 'delta_acc_max':  round(d_acc.max(), 4),
                     'best_layer_prob': int(np.argmax(d_gp))  + 1, 'delta_prob_max': round(d_gp.max(), 4),
                     'conv_layer': conv})
    return pd.DataFrame(rows).sort_values('support', ascending=False).reset_index(drop=True)


impact_summary = {}
for task, min_sup in (('upos', 30), ('deprel', 30), ('head', 100)):
    impact_summary[task] = summarize_layer_impact(perlabel_stats[task], min_support=min_sup)
    impact_summary[task].to_csv(f'./layerwise_impact_summary_{task}_biaffine.csv', index=False)

print('Média (ponderada por support) da camada mais impactante:')
for task in ('upos', 'deprel', 'head'):
    s = impact_summary[task]
    w_prob = np.average(s['best_layer_prob'], weights=s['support'])
    w_conv = np.average(s['conv_layer'].astype(float), weights=s['support'])
    print(f"  {task:7s} → impacto: camada {w_prob:.1f} | convergência: camada {w_conv:.1f}")
display(impact_summary['upos'])
display(impact_summary['deprel'].head(20))


## Early exit (biaffine): usar apenas as melhores camadas é suficiente?

Coletamos as predições de todas as 13 camadas em uma passada e avaliamos qualquer
combinação offline — acurácias, **UAS** e **LAS** (deprel avaliado na head predita
pela própria camada, como no forward).

In [ ]:
def collect_predictions_all_layers(sentences, gold_upos_list, gold_deprel_list, gold_head_list,
                                   model, tokenizer, device=None, max_sentences=None):
    """Uma passada: argmax de cada camada para cada tarefa (versão biaffine)."""
    if device is None:
        device = next(model.parameters()).device
    model.eval()

    preds_store, golds_store = [], []
    n = len(sentences) if max_sentences is None else min(max_sentences, len(sentences))
    for i in tqdm(range(n)):
        tokens = sentences[i]
        golds = {
            'upos':   [UPOS_LABELS_TO_IDX[u]   for u in gold_upos_list[i]],
            'deprel': [DEPREL_LABELS_TO_IDX[d] for d in gold_deprel_list[i]],
            'head':   [int(h)                  for h in gold_head_list[i]],
        }
        inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                           padding=True, truncation=True).to(device)
        with torch.no_grad():
            bert_out = model.bert(
                input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'],
                token_type_ids=inputs.get('token_type_ids'), output_hidden_states=True,
            )
            hs_all = torch.stack(bert_out.hidden_states, dim=0)[:, 0]
            logits_upos, logits_head, logits_deprel = biaffine_heads_from_hidden(model, hs_all)

            word_ids = inputs.word_ids(batch_index=0)
            first_subs, tok_idxs = [], []
            seen = set()
            for pos, w in enumerate(word_ids):
                if w is not None and w not in seen and w < len(tokens):
                    seen.add(w)
                    first_subs.append(pos)
                    tok_idxs.append(w)
            if not first_subs:
                continue
            fs = torch.tensor(first_subs, device=device)

            sent_preds = {
                'upos':   logits_upos[:, fs].argmax(-1).cpu().numpy().astype(np.int16),
                'deprel': logits_deprel[:, fs].argmax(-1).cpu().numpy().astype(np.int16),
                'head':   logits_head[:, fs].argmax(-1).cpu().numpy().astype(np.int16),
            }
            sent_golds = {t: np.array([golds[t][k] for k in tok_idxs], dtype=np.int16)
                          for t in ('upos', 'deprel', 'head')}
            preds_store.append(sent_preds)
            golds_store.append(sent_golds)
    return preds_store, golds_store


def eval_layer_combo(preds_store, golds_store, upos_layer, deprel_layer, head_layer):
    """Acurácias + UAS/LAS para uma combinação de camadas (uma por tarefa)."""
    total = upos_c = deprel_c = uas_c = las_c = 0
    for preds, golds in zip(preds_store, golds_store):
        p_upos, p_deprel, p_head = preds['upos'][upos_layer], preds['deprel'][deprel_layer], preds['head'][head_layer]
        g_upos, g_deprel, g_head = golds['upos'], golds['deprel'], golds['head']
        total    += len(g_upos)
        upos_c   += (p_upos == g_upos).sum()
        deprel_c += (p_deprel == g_deprel).sum()
        head_ok   = (p_head == g_head)
        uas_c    += head_ok.sum()
        las_c    += (head_ok & (p_deprel == g_deprel)).sum()
    return {'upos_layer': upos_layer, 'deprel_layer': deprel_layer, 'head_layer': head_layer,
            'upos_acc': upos_c / total, 'deprel_acc': deprel_c / total,
            'uas': uas_c / total, 'las': las_c / total}


preds_store, golds_store = collect_predictions_all_layers(
    test_sentences, test_upos, test_deprel, test_head,
    layerwise_model, layerwise_tokenizer,
)

n_layers_total = preds_store[0]['upos'].shape[0]
per_layer_acc = pd.DataFrame([
    eval_layer_combo(preds_store, golds_store, L, L, L) for L in range(n_layers_total)
]).rename(columns={'upos_layer': 'layer'}).drop(columns=['deprel_layer', 'head_layer'])
per_layer_acc.to_csv('./layerwise_per_layer_metrics_biaffine.csv', index=False)

FINAL = n_layers_total - 1
configs = {
    'baseline: tudo na camada final (12)': (FINAL, FINAL, FINAL),
    'early exit na camada 11':             (11, 11, 11),
    'early exit na camada 10':             (10, 10, 10),
    'early exit na camada 9':              (9, 9, 9),
    'early exit na camada 8':              (8, 8, 8),
}
rows = []
for name, (lu, ld, lh) in configs.items():
    r = eval_layer_combo(preds_store, golds_store, lu, ld, lh)
    r['config'] = name
    rows.append(r)
comparison = pd.DataFrame(rows)[
    ['config', 'upos_layer', 'deprel_layer', 'head_layer', 'upos_acc', 'deprel_acc', 'uas', 'las']
]
comparison.to_csv('./layerwise_early_exit_comparison_biaffine.csv', index=False)
per_layer_acc.round(4)


## Block skip (biaffine): é possível ativar apenas a camada 12?

Embeddings entram diretamente em blocos escolhidos do encoder; as cabeças biaffine
leem a saída. Sanity check: todos os blocos == inferência normal.

In [ ]:
def eval_block_subset(blocks, sentences, gold_upos_list, gold_deprel_list, gold_head_list,
                      model, tokenizer, device=None, max_sentences=None, desc=None):
    """Passa embeddings apenas pelos blocos indicados e avalia as cabeças biaffine."""
    if device is None:
        device = next(model.parameters()).device
    model.eval()

    encoder_layers = model.bert.encoder.layer
    total = upos_c = deprel_c = uas_c = las_c = 0

    n = len(sentences) if max_sentences is None else min(max_sentences, len(sentences))
    for i in tqdm(range(n), desc=desc):
        tokens = sentences[i]
        golds = {
            'upos':   np.array([UPOS_LABELS_TO_IDX[u]   for u in gold_upos_list[i]], dtype=np.int64),
            'deprel': np.array([DEPREL_LABELS_TO_IDX[d] for d in gold_deprel_list[i]], dtype=np.int64),
            'head':   np.array([int(h)                  for h in gold_head_list[i]], dtype=np.int64),
        }
        inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                           padding=True, truncation=True).to(device)
        with torch.no_grad():
            hs = model.bert.embeddings(
                input_ids=inputs['input_ids'],
                token_type_ids=inputs.get('token_type_ids'),
            )
            for b in blocks:
                hs = encoder_layers[b](hs)
                if isinstance(hs, tuple):
                    hs = hs[0]

            logits_upos, logits_head, logits_deprel = biaffine_heads_from_hidden(model, hs)

            word_ids = inputs.word_ids(batch_index=0)
            first_subs, tok_idxs = [], []
            seen = set()
            for pos, w in enumerate(word_ids):
                if w is not None and w not in seen and w < len(tokens):
                    seen.add(w)
                    first_subs.append(pos)
                    tok_idxs.append(w)
            if not first_subs:
                continue
            fs = torch.tensor(first_subs, device=device)

            p_upos   = logits_upos[0, fs].argmax(-1).cpu().numpy()
            p_deprel = logits_deprel[0, fs].argmax(-1).cpu().numpy()
            p_head   = logits_head[0, fs].argmax(-1).cpu().numpy()

        g_upos, g_deprel, g_head = (golds[t][tok_idxs] for t in ('upos', 'deprel', 'head'))
        total    += len(tok_idxs)
        upos_c   += (p_upos == g_upos).sum()
        deprel_c += (p_deprel == g_deprel).sum()
        head_ok   = (p_head == g_head)
        uas_c    += head_ok.sum()
        las_c    += (head_ok & (p_deprel == g_deprel)).sum()

    blocos_str = ','.join(str(b + 1) for b in blocks) if blocks else 'nenhum (embeddings)'
    return {'blocos_ativos': blocos_str, 'n_blocos': len(blocks),
            'upos_acc': upos_c / total, 'deprel_acc': deprel_c / total,
            'uas': uas_c / total, 'las': las_c / total}


block_configs = {
    'sanity: todos os blocos (== inferência normal)': list(range(12)),
    'APENAS bloco 12':                                [11],
    'APENAS bloco 12, aplicado 12x':                  [11] * 12,
    'blocos 11 e 12':                                 [10, 11],
    'blocos 9 a 12':                                  [8, 9, 10, 11],
    'blocos 7 a 12 (pula metade inferior)':           [6, 7, 8, 9, 10, 11],
    'nenhum bloco (embeddings puras)':                [],
}
block_rows = []
for name, blocks in block_configs.items():
    r = eval_block_subset(blocks, test_sentences, test_upos, test_deprel, test_head,
                          layerwise_model, layerwise_tokenizer, desc=name)
    r['config'] = name
    block_rows.append(r)
block_comparison = pd.DataFrame(block_rows)[
    ['config', 'blocos_ativos', 'n_blocos', 'upos_acc', 'deprel_acc', 'uas', 'las']
]
block_comparison.to_csv('./layerwise_block_skip_comparison_biaffine.csv', index=False)
block_comparison.round(4)
